In [11]:
from pathlib import Path
import json
import pandas as pd

BASE_DIR = Path(".").resolve()
MAPPINGS_DIR = BASE_DIR / "mappings_stx"


In [15]:

rows = []

for fp in sorted(MAPPINGS_DIR.glob("*.json")):
    firm_id = fp.stem
    data = json.loads(fp.read_text(encoding="utf-8"))

    for v in data.get("variables", []):
        if v.get("needs_manual_review", False):
            final_choice = v.get("final_choice", [])
            # make final_choice readable
            final_str = "; ".join([f"{x['sheet_name']}::{x['row_label']}" for x in final_choice]) if final_choice else ""
            rows.append({
                "firm_id": firm_id,
                "variable": v.get("variable", ""),
                "notes": v.get("notes", ""),
                "final_choice": final_str,
                "num_candidates": len(v.get("candidates", [])),
            })

if len(rows) == 0:
    print("No manual reviews needed. All mappings have been reviewed and finalized.")
else:
    df_review = pd.DataFrame(rows).sort_values(["variable", "firm_id"]).reset_index(drop=True)

    print(f"Manual review needed for {len(df_review)} firm-variable mappings across {df_review['firm_id'].nunique() if len(df_review) else 0} firms.")
    display(df_review)

Manual review needed for 242 firm-variable mappings across 131 firms.


,firm_id,variable,notes,final_choice,num_candidates
0,AAK.ST,BE,Preferred owner equity excluding non-controlli...,Balance Sheet::Shareholders equity,2
1,ABB.ST,BE,No BE candidates were provided. Selected 'Tota...,Balance Sheet::Total ABB stockholders equity,0
2,ACELP.ST,BE,Preferred candidate list missed the better own...,Balance Sheet::Equity attributable to Parent C...,3
3,ACRIa.ST,BE,"Candidate list includes 'Total Equity', but 'S...",Balance Sheet::Shareholders Equity,1
4,ADDTb.ST,BE,Candidate list misses the preferred owner/pare...,Balance Sheet::Equity attributable to Parent C...,2
...,...,...,...,...,...
237,MEABb.ST,XSGA_COMPONENTS,Candidate list appears incomplete for SG&A. No...,Income Statement::Personnel Expenses; Income S...,1
238,MEKO.ST,XSGA_COMPONENTS,No explicit SG&A total row is present. Used ov...,Income Statement::Personnel expenses; Income S...,2
239,MOMENT.ST,XSGA_COMPONENTS,No explicit SG&A total row is present. Used ov...,Income Statement::Personnel costs; Income Stat...,2
240,MSABb.ST,XSGA_COMPONENTS,Used rows outside the candidate list because t...,Income Statement::Selling and Administration E...,1


In [16]:
df_review = pd.DataFrame(rows)

df_xsga = (
    df_review[df_review["variable"] == "XSGA_COMPONENTS"]
    .sort_values(["firm_id"])
    .reset_index(drop=True)
)

print(f"XSGA_COMPONENTS manual reviews: {len(df_xsga)} rows across {df_xsga['firm_id'].nunique() if len(df_xsga) else 0} firms.")
display(df_xsga)

XSGA_COMPONENTS manual reviews: 91 rows across 91 firms.


,firm_id,variable,notes,final_choice,num_candidates
0,8TRA.ST,XSGA_COMPONENTS,Candidate list appears incomplete for SG&A bec...,Income Statement::Distribution expenses; Incom...,3
1,AAK.ST,XSGA_COMPONENTS,No explicit SG&A subtotal is present. Used ove...,Income Statement::Other external expenses; Inc...,1
2,ACADE.ST,XSGA_COMPONENTS,No explicit SG&A subtotal row is present. Cand...,Income Statement::Personnel expenses; Income S...,2
3,ACAST.ST,XSGA_COMPONENTS,Candidate list is empty. No explicit single SG...,Income Statement::Sales and marketing costs; I...,0
4,ACELP.ST,XSGA_COMPONENTS,"Candidate list was empty, so rows were chosen ...",Income Statement::Administrative costs; Income...,4
...,...,...,...,...,...
86,MEABb.ST,XSGA_COMPONENTS,Candidate list appears incomplete for SG&A. No...,Income Statement::Personnel Expenses; Income S...,1
87,MEKO.ST,XSGA_COMPONENTS,No explicit SG&A total row is present. Used ov...,Income Statement::Personnel expenses; Income S...,2
88,MOMENT.ST,XSGA_COMPONENTS,No explicit SG&A total row is present. Used ov...,Income Statement::Personnel costs; Income Stat...,2
89,MSABb.ST,XSGA_COMPONENTS,Used rows outside the candidate list because t...,Income Statement::Selling and Administration E...,1
